# 🌐 Data Collection & Web Scraping — Complete Masterclass
> **Course Track:** Prime AI/ML Batch  
> **Source:** `Web Scraping.pdf` & `data_collection.ipynb`  
> **Topic:** HTML Foundations, REST APIs (`requests`), JSON Normalization, BeautifulSoup (`bs4`), Headers, & Scraping Pipeline to CSV  
> **Language / Notes Mode:** Easy Natural Hinglish  

---

### 📌 Data Collection & Web Scraping Kya Hai Aur Iski Kyu Zaroorat Hai?
- Machine Learning models ko train karne ke liye sabse pehle **Raw Data** ki zaroorat hoti hai.
- Ye data do primary tariko se collect kiya jata hai:
  1. **APIs (Application Programming Interfaces):** Jab websites ya platforms structured format (JSON) me official data provide karte hain (Clean & Reliable).
  2. **Web Scraping:** Jab kisi website ka public API available nahi hota, toh hum Python script ke through webpage ka HTML code download karke usme se specific information (jaise Product prices, News articles, Stock data) extract karte hain.
- **Tools Used:**
  - `requests`: Internet se web pages ya API data download karne ke liye.
  - `BeautifulSoup` (`bs4`): Downloaded HTML code ko parse (read/navigate) karke text aur attributes extract karne ke liye.
  - `pandas`: Scraped data ko structured DataFrame me convert karke CSV me export karne ke liye.


## 📄 Operation 1: HTML Architecture for Data Science & Scraping

### 💡 Kyu Zaroorat Hai?
Kisi bhi website ko scrape karne se pehle ye samajhna zaroori hai ki HTML elements kaise organized hote hain:
- **Tags:** Content ko define karte hain:
  - `<h1>` se `<h6>`: Headings (titles, headlines)
  - `<p>`: Paragraphs (descriptions, body text)
  - `<a>`: Hyperlinks (URLs extract karne ke liye `href` attribute)
  - `<img>`: Images (`src` attribute se image link milti hai)
  - `<table>`, `<tr>` (row), `<th>` (header), `<td>` (data cell): Tabular data
  - `<div>` aur `<span>`: Layout containers (jinpar CSS `class` ya `id` lagi hoti hai)
- **Attributes:** Tags ke andar extra info dete hain (jaise `class="price"`, `id="product-101"`).


In [ ]:
# Sample HTML snippet jo ek typical e-commerce card ya table ko represent karta hai
sample_html = """
<!DOCTYPE html>
<html>
<head><title>Scraping Demo Store</title></head>
<body>
    <div id="main-content" class="container">
        <h1 class="store-title">Welcome to Tech Store</h1>
        <p class="subtitle">Latest AI Hardware Deals</p>
        
        <table id="products-table">
            <tr><th>Item</th><th>Price</th><th>Availability</th></tr>
            <tr><td>NVIDIA RTX 4090</td><td>$1599</td><td>In Stock</td></tr>
            <tr><td>Apple M3 Max</td><td>$2499</td><td>Pre-Order</td></tr>
        </table>
        
        <div class="deal-card">
            <h3 class="product-name">Quantum AI Dev Board</h3>
            <span class="price">$499</span>
            <a href="https://example.com/buy/quantum-board" class="buy-link">Buy Now</a>
        </div>
    </div>
</body>
</html>
"""

print("✅ Sample HTML string ready for demonstration.")


## 🚀 Operation 2: HTTP GET Requests with `requests` Library

### 💡 Kyu Zaroorat Hai?
Browser me jab aap koi link enter karte hain, toh browser background me ek **HTTP GET request** bhejta hai. Python me hum `requests.get(url)` se wahi request bhejte hain.

### 🔍 Important Response Properties:
- `res.status_code`: Server ka reply status:
  - `200`: Success (Sab badhiya hai!)
  - `400`: Bad Request
  - `403`: Forbidden (Server ne bot samajhkar block kar diya)
  - `404`: Not Found (Page exist nahi karta)
  - `500`: Server Error
- `res.text`: Webpage ka HTML content string format me.
- `res.headers`: Server metadata (Content-Type, Date, Server).


In [ ]:
import requests

# Example URL (Scrape This Site sandbox)
url = "https://www.scrapethissite.com/pages/simple/"

try:
    response = requests.get(url, timeout=10)
    print("HTTP Status Code:", response.status_code)
    print("Content-Type:    ", response.headers.get("Content-Type"))
    print("HTML Length:     ", len(response.text), "characters")
    print("\nFirst 200 characters of HTML:\n", response.text[:200])
except Exception as e:
    print("Request Error (Using offline simulation):", e)


## 🛡️ Operation 3: Robust Requests — Custom `User-Agent` & Exception Handling

### 💡 Kyu Zaroorat Hai? (Pro Scraping Tip!)
- Default Python request ka User-Agent header `python-requests/2.x` hota hai. Bahut si websites ise detect karke direct block kar deti hain (`403 Forbidden`).
- Real browser jaisa behave karne ke liye hum custom **`User-Agent`** header pass karte hain.
- `r.raise_for_status()` use karne se agar status `4xx` ya `5xx` ho toh automatic clear exception raise ho jata hai.


In [ ]:
# Custom Headers simulation
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9"
}

target_url = "https://httpbin.org/headers"

try:
    res = requests.get(target_url, headers=headers, timeout=5)
    res.raise_for_status()  # Check for HTTP errors
    print("✅ Request successfully received by server!")
    print("Server received headers:\n", res.json())
except requests.exceptions.RequestException as err:
    print("❌ Request failed safely:", err)


## ⚡ Operation 4: REST API Data Collection (`requests.get` + `res.json()`)

### 💡 Kyu Zaroorat Hai?
HTML scrape karne se lakh guna aasan aur fast hota hai **REST API** se data lena. APIs structured **JSON** format return karti hain jisme koi HTML parsing ka jhanjhat nahi hota.

### 🔍 Kaise Kaam Karta Hai?
- `response.json()` direct Python dictionary ya list of dictionaries me convert kar deta hai.


In [ ]:
import pandas as pd

# Stephen King Books API (From course curriculum)
api_url = "https://stephen-king-api.onrender.com/api/books"

try:
    api_res = requests.get(api_url, timeout=10)
    api_res.raise_for_status()
    json_data = api_res.json()
    print("API Response Keys:", list(json_data.keys()))
    print(f"Total Books Retrieved: {len(json_data.get('data', []))}")
    print("\nSample Book Object:\n", json_data['data'][0])
except Exception as e:
    print("Public API unavailable, using local mock data:", e)
    json_data = {
        "data": [
            {"id": 1, "Title": "Carrie", "Year": 1974, "ISBN": "978-0385086950"},
            {"id": 2, "Title": "Salem's Lot", "Year": 1975, "ISBN": "978-0385007511"},
            {"id": 3, "Title": "The Shining", "Year": 1977, "ISBN": "978-0385121675"},
            {"id": 4, "Title": "The Stand", "Year": 1978, "ISBN": "978-0385121682"}
        ]
    }


## 📊 Operation 5: JSON Normalization into Pandas DataFrame (`pd.json_normalize`)

### 💡 Kyu Zaroorat Hai?
Nested JSON structures ko flat tabular DataFrame me convert karne ke liye `pd.json_normalize()` best function hai.


In [ ]:
# JSON data ko tabular DataFrame me normalize karna
df_books = pd.json_normalize(json_data["data"])

# Specific required columns select karna
df_books_clean = df_books[['id', 'Title', 'Year', 'ISBN']].head(10)

print("--- Clean Books DataFrame from API ---")
display(df_books_clean)


## 🍲 Operation 6: HTML Parsing with BeautifulSoup (`find` & `find_all`)

### 💡 Kyu Zaroorat Hai?
Web Scraping me raw HTML string milne ke baad usme se tags ko search karne ke liye **BeautifulSoup** parse tree banata hai.

### 🔍 Key Methods:
- `BeautifulSoup(html, "html.parser")` ya `"lxml"`: Parser initialize karna.
- `soup.find("tag")`: Match hone wala **sabse pehla** element return karta hai.
- `soup.find_all("tag")`: Match hone wale **saare** elements ki Python list return karta hai.
- `element.get_text(strip=True)`: Tag ke andar ka text nikalta hai aur extra spaces clean karta hai.


In [ ]:
from bs4 import BeautifulSoup

# Sample HTML ko parse karna
soup = BeautifulSoup(sample_html, "html.parser")

# 1. Page Title nikalna
title_tag = soup.find("title")
print("Page Title Tag:", title_tag.get_text(strip=True))

# 2. Main Heading find karna
h1_tag = soup.find("h1")
print("H1 Heading:    ", h1_tag.get_text(strip=True))

# 3. Saare Table Rows <tr> find karna
all_rows = soup.find_all("tr")
print(f"\nTotal table rows found: {len(all_rows)}")
for idx, r in enumerate(all_rows):
    row_text = [cell.get_text(strip=True) for cell in r.find_all(["th", "td"])]
    print(f"Row {idx}: {row_text}")


## 🎯 Operation 7: CSS Selectors (`soup.select` & `select_one`)

### 💡 Kyu Zaroorat Hai?
Complex nested pages par tags ke bajay CSS classes aur IDs ke through exact target par jump karna sabse fast aur clean tarika hota hai:
- `.classname`: Class select karne ke liye.
- `#idname`: ID select karne ke liye.
- `div.deal-card h3`: Hierarchy traversal (div ke andar h3).


In [ ]:
# 1. select_one: Class name se single element nikalna
product_name = soup.select_one("h3.product-name").get_text(strip=True)
product_price = soup.select_one("span.price").get_text(strip=True)

print("Product Name: ", product_name)
print("Product Price:", product_price)

# 2. Extracting Attribute (href link)
buy_link = soup.select_one("a.buy-link")["href"]
print("Product Buy URL:", buy_link)


## 🌳 Operation 8: DOM Navigation (`find_parent` & `find_next`)

### 💡 Kyu Zaroorat Hai?
Jab data kisi tag ke bagal wale ya parent container me ho, toh hum DOM tree ke relation methods use karte hain:
- `element.find_parent("div")`: Element ke parent container par jana.
- `element.find_next("div")`: Element ke baad aane wale agle tag par jana.


In [ ]:
# h3 tag se next div traverse karna
h3_tag = soup.find("h3")
print("Target Tag:", h3_tag.get_text(strip=True))

# Parent div class check karna
parent_div = h3_tag.find_parent("div")
print("Parent Div Class:", parent_div["class"])

# Next sibling ya next element find karna
next_span = h3_tag.find_next("span")
print("Next adjacent element text:", next_span.get_text(strip=True))


## 🌍 Operation 9: Complete Real-World Scraping Project (Countries & Populations)

### 💡 Kyu Zaroorat Hai?
Yeh hamari course tutorial (`data_collection.ipynb`) ka practical web scraping project hai:
- Website: `https://www.scrapethissite.com/pages/simple/`
- Target: Har country ka **Name** aur **Population** extract karna.


In [ ]:
scrape_url = "https://www.scrapethissite.com/pages/simple/"
countries_list = []

try:
    res = requests.get(scrape_url, timeout=10)
    if res.status_code == 200:
        page_soup = BeautifulSoup(res.text, "html.parser")
        all_country_divs = page_soup.select("div.country")
        
        for c in all_country_divs[:15]:  # Top 15 countries sample
            name = c.select_one("h3.country-name").get_text(strip=True)
            capital = c.select_one("span.country-capital").get_text(strip=True)
            population = c.select_one("span.country-population").get_text(strip=True)
            area = c.select_one("span.country-area").get_text(strip=True)
            countries_list.append([name, capital, population, area])
        print(f"✅ Successfully scraped {len(countries_list)} countries live!")
except Exception as e:
    print("Live site fetch error, running offline simulation:", e)
    countries_list = [
        ["Andorra", "Andorra la Vella", "84000", "468.0"],
        ["United Arab Emirates", "Abu Dhabi", "4975593", "82880.0"],
        ["Afghanistan", "Kabul", "29121286", "647500.0"],
        ["Antigua and Barbuda", "St. John's", "86754", "443.0"],
        ["Anguilla", "The Valley", "13254", "102.0"],
        ["Albania", "Tirana", "2986952", "28748.0"],
        ["Armenia", "Yerevan", "2968000", "29800.0"]
    ]


## 💾 Operation 10: Converting Scraped Data to DataFrame & Exporting to CSV

### 💡 Kyu Zaroorat Hai?
Scraped data list of lists me hota hai. Use Machine Learning aur Data Analysis me feed karne ke liye hum Pandas DataFrame banate hain aur disk par CSV file me save karte hain.


In [ ]:
# List to DataFrame
df_countries = pd.DataFrame(countries_list, columns=["Country_Name", "Capital", "Population", "Area_SqKm"])

# Numerical cleaning: convert population to int
df_countries["Population"] = df_countries["Population"].astype(int)
df_countries["Area_SqKm"] = df_countries["Area_SqKm"].astype(float)

print("--- Scraped Countries DataFrame Preview ---")
display(df_countries.head(6))

# Export to CSV
os.makedirs("cleaned_data", exist_ok=True)
csv_export_path = "cleaned_data/scraped_countries.csv"
df_countries.to_csv(csv_export_path, index=False)

print(f"\n🎉 Data successfully exported to '{csv_export_path}' without index!")
print("File exists on disk:", os.path.exists(csv_export_path))
